In [ ]:
import pandas as pd

# File placed manually (Colab)
df = pd.read_csv("/content/presales_data_sample.csv",
                 dtype={"main_postcode": str, "main_street_number": str, "input_row_key": str,
                        "input_main_postcode": str, "input_main_street_number": str})

# Check if we have fully duplicate rows
print("Duplicate rows: ", df.duplicated().any())
df = df.drop_duplicates()

only_numeric = df['input_row_key'].dropna().astype(str).str.isdigit().all()
print("Input row key has only numerical values: ", only_numeric)
# Checked PowerQuery vs Pandas: the key is numeric only, so the text was a display problem in PQ

# Start EDA - basic info
print("Object columns: ")
display(df.describe(include='object'))

print("Numerical columns: ")
display(df.describe())


In [ ]:
# Split input based on column name
input_df = df.loc[:, df.columns.str.startswith("input_")].copy()

# 1:many relationship with output
input_df = input_df.drop_duplicates().reset_index(drop=True)

# Check why we have same company name twice
display(input_df['input_company_name'].value_counts())

# Identified duplicated APPLE DISTRIBUTION INTERNATIONAL LIMITED
display(input_df[input_df['input_company_name'] == "APPLE DISTRIBUTION INTERNATIONAL LIMITED"])
# For row_key 44 and 45 we have the same company name, country and country code
# unmatched columns are only row key (expected) and nulls -> most likely duplicate

output_columns = ['input_row_key','veridion_id', 'company_name', 'company_legal_names',
       'company_commercial_names', 'main_country_code', 'main_country',
       'main_region', 'main_city', 'main_postcode', 'main_street',
       'main_street_number', 'locations']
output_df = df[output_columns].copy()

# Check output for input row 44 and 45
output_df[output_df['input_row_key'].isin(["44","45"])]

# The correct match is for row 45 (where we have more data)

### Findings

input_row_key 44 and 45 have the same company and the same data, except that some values are null in one of them. This will be from my point of view a cantidate of duplicate row.
The good news: our correct candidate appears in the output for both rows.

What to do next? We have two drastic options:

- **Drop one**
  - *Advantage:* less duplicated work, row with more data is more likely to rank the match correctly
  - *Disadvantage:* we can't be sure a null means the same as the value in the other row (e.g. a null region in 44 does not automatically mean Munster, as in 45). Additionaly we will lose a row from the output
- **Keep both**
  - *Advantage:* we don't discard data when we are not sure - only if fully duplicated (already did)
  - *Disadvantage:* the row with less data has less to compare on, so its match may score lower or less clearly - in this case I observed the closest one for both match the additional data from 45

**Decision:** I choose the one in between: only flag possible duplicates and don't remove anything. This way we can decide later

*Possible duplicate* = same company name and no column with two different non-null values.
This runs after cleaning, because small differences in the names may hide other duplicates

**Recommendation:** If we decide to remove one I will keep the row with fewer nulls.

In [ ]:
""" AI generated function based on my rules
Rules:
    1. Keep keys intact (handled in clean_df)
    2. Lower, strip and remove accents
    3. 2 ways to remove punctuation: one for company name, one for the rest
       - name: remove . and / without space ("a/s" -> "as"), then remove legal words
       - rest: punctuation -> space, then remove place words
    4. Additional words (legal / place) are sometimes missing from the corresponding match
    5. Empty or null -> None
"""
import re, unicodedata
import pandas as pd

KEYS = {"input_row_key", "veridion_id", "unique_key"}
LEGAL = r"\b(corporation|corp|company|co|limited|ltd|srl|sa|as|ab|oy|ag|bv|nv|spa|gmbh|llc|inc)\b"
PLACE = r"\b(municipality|region|city|district|province|county|island)\b"

def clean_text(value, is_name=False):
    if pd.isna(value):
        return None
    s = unicodedata.normalize("NFKD", str(value).casefold())
    s = "".join(c for c in s if not unicodedata.combining(c))  # accents: ä -> a
    if is_name:
        s = re.sub(r"[./]", "", s)                 # "a/s" -> "as", "s.a." -> "sa"
    s = re.sub(r"[^\w\s]|_", " ", s)               # other punctuation -> space
    s = re.sub(LEGAL if is_name else PLACE, " ", s) # legal words only in names, place words only elsewhere
    return re.sub(r"\s+", " ", s).strip() or None

def clean_df(df):
    return df.apply(lambda c: c.map(lambda v: clean_text(v, "company_name" in c.name))
                    if c.dtype == "object" and c.name not in KEYS else c)


In [ ]:
input_df = clean_df(input_df)
input_df.head()

In [ ]:
# Check after cleaning how many duplicate company names we have
display(input_df["input_company_name"].value_counts())

# As expected, now we have more <- let's make sure other non-null columns are the same
cols_no_nulls = [col for col in input_df.columns
    if col != "input_row_key" and input_df[col].notna().all()]

duplicate_idx = []
for name, group in input_df.groupby(cols_no_nulls):
    if len(group) > 1 and not (group.drop(columns="input_row_key").nunique() > 1).any():
        duplicate_idx += list(group.index)

# Flag those columns
input_df["possible_duplicate"] = None
input_df.loc[duplicate_idx, "possible_duplicate"] = "Possible duplicate"

display(input_df[input_df["possible_duplicate"].notna()].head())
display(input_df.head())

## Output cleaning and preprocessing

In [ ]:
# Match input_company_name with legal names and commercial names as well -> unpivot columns to have one column
print("Before: ", len(output_df))
columns_unpivot = ['company_name', 'company_legal_names','company_commercial_names']
output_names_df = output_df.melt(
      id_vars=output_df.columns.difference(columns_unpivot),
      value_vars= columns_unpivot,
      var_name='company_title_type',
      value_name='company_name_det'
)
print("After unpivot: ", len(output_names_df))
# We know from PQ analysis that commercial names can hold several names separated by |
output_names_df["company_name_det"] = output_names_df["company_name_det"].str.split("|")
output_names_df = output_names_df.explode("company_name_det")
print("After | split", len(output_names_df))

# Basic cleaning and drop null/empty
output_names_df["company_name_det"] = output_names_df["company_name_det"].str.strip()
output_names_df = output_names_df[output_names_df["company_name_det"].notna() & (output_names_df["company_name_det"] != "")]
print("Final company names version: ", len(output_names_df))

In [ ]:
"""
AI generated based on my rule:
  Check if we have any pipes left (combined data worth checking)
  List of columns -> if None we can clean
"""

def check_no_pipes(df):
    found = df.select_dtypes(include="object").apply(
        lambda col: col.str.contains("|", regex=False, na=False).any()
    )

    return found[found].index.tolist() or None

In [ ]:
len(input_df)

In [ ]:
# After analysis seems like locations column could be used to match the two dfs
locations_df = output_names_df[['input_row_key', 'veridion_id', 'company_name_det', 'locations', 'company_title_type']].dropna(subset=['locations'])

locations_df = locations_df.assign(locations=locations_df['locations'].str.split('|')).explode('locations') # <- multiple locations: delimitor '|'
locations_df['locations'] = locations_df['locations'].str.strip()
locations_df = locations_df[locations_df['locations'] != ''].drop_duplicates().reset_index(drop=True)  # dedupe and remove empty
print("Location rows:", len(locations_df))

location_details = locations_df['locations'].str.split(', ', expand=True).iloc[:, :7]
location_details.columns = ['main_country_code', 'main_country', 'main_region', 'main_city',
                            'main_postcode', 'main_street', 'main_street_number']

locations_df = locations_df.drop(columns='locations').join(location_details)

#Check if we have any columns that should be split
pipe_cols = check_no_pipes(locations_df)

if pipe_cols:
    print(f"Pipes found in: {pipe_cols}")
else:
    locations_df = clean_df(locations_df).drop_duplicates()  # Clean to the location and drop duplicates

In [ ]:
output_names_df = output_names_df.drop(columns=["locations"], errors="ignore")

pipe_cols_input = check_no_pipes(input_df)
pipe_cols_output = check_no_pipes(output_names_df)

if pipe_cols_output:
    print(f"Pipes found OUTPUT in: {pipe_cols_output}")
else:
  input_df = clean_df(input_df)

if pipe_cols_input:
    print(f"Pipes found INPUT in: {pipe_cols_input}")
else:
    output_names_df = clean_df(output_names_df)

In [ ]:
output_df = pd.concat([locations_df,output_names_df], ignore_index=True)
output_df = output_df.drop_duplicates().reset_index(drop=True)

output_df.head()

## Splink implementation

In [ ]:
# Prepare data for Splink

# Matching columns should have the same names
print("Before match:")
print("Output: ",sorted(output_df.columns))
print("Input: ",sorted(input_df.columns))

input_df.columns = input_df.columns.str.replace("input_", "")
output_df = output_df.rename(columns = {"input_row_key": "row_key", "company_name_det": "company_name"})

print("\nAfter match:")
print(sorted(input_df.columns))

# Creating/Renaming the unique_id
print("\nInput unique column: ",input_df.columns[input_df.nunique() == input_df.notna().sum()].tolist())
print("Output unique column: ",output_df.columns[output_df.nunique() == output_df.notna().sum()].tolist())

input_df["unique_id"] = input_df["row_key"]
output_df["unique_id"] = range(len(output_df))
output_df["unique_id"] = output_df["unique_id"].astype(str)
output_df.head()

In [ ]:
%pip install splink

In [ ]:
from splink import DuckDBAPI
from splink.exploratory import completeness_chart

db_api = DuckDBAPI()
print("Input:")
input_df_sdf = db_api.register(input_df)
display(completeness_chart(input_df_sdf, table_names_for_chart = "input"))

print("Output:")
output_df_sdf = db_api.register(output_df)
display(completeness_chart(output_df_sdf, table_names_for_chart = "output"))

In [ ]:
from splink.exploratory import profile_columns

profile_columns(input_df_sdf)

Decided SQL

In [ ]:
input_df

In [ ]:
# Rules for the name comparison (name_cmp):
# Remove the row's own country (main_country) from its name, only if it appears as a whole word
# Same rule on input and output.
# If the name is only the country, keep the original name - it should not be the case, but just to be sure because we are eliminating other words as well in clean data.
# Reason: a country word shared by both names is not evidence of the same company (the country is already compared on its own), and it would artificially increase any similarity score.

def remove_country(name, country):
    if pd.isna(name) or pd.isna(country) or not str(country).strip():
        return name
    s = re.sub(rf"\b{re.escape(str(country))}\b", " ", str(name))
    s = re.sub(r"\s+", " ", s).strip()
    return s or name

input_df['company_name'] = [remove_country(n, c) for n, c in zip(input_df['company_name'], input_df['main_country'])]
output_df['company_name'] = [remove_country(n, c) for n, c in zip(output_df['company_name'], output_df['main_country'])]

In [ ]:
import duckdb

In [ ]:
NAME_SIMILARITY_THRESHOLD = 70 # Under what % we should eliminate this from match

q = f"""
WITH match AS (
SELECT
  i.row_key,
  veridion_id,
  i.company_name,
  o.company_name,

  -- If country/region not matching -> Not a match
  COALESCE((i.main_region = o.main_region)::INT, 0)
  + COALESCE((i.main_city = o.main_city)::INT, 0)
  + COALESCE((i.main_postcode = o.main_postcode)::INT, 0)
  + COALESCE((i.main_street = o.main_street)::INT, 0)
  + COALESCE((i.main_street_number = o.main_street_number)::INT, 0) as s_match,

  -- Get similarity score if both names are not null
  CASE WHEN i.company_name IS NULL OR o.company_name IS NULL THEN 0
      ELSE jaro_winkler_similarity(i.company_name, o.company_name) END AS name_score

FROM input_df i
  JOIN output_df o
  ON i.row_key=o.row_key
  AND i.main_country_code = o.main_country_code
),

scores AS (
SELECT
  row_key,
  veridion_id,
  MAX((s_match)/5)*100 as perc_match,
  MAX(name_score)*100 as max_name_score,
  CASE
    WHEN max_name_score > {NAME_SIMILARITY_THRESHOLD}
    THEN 0.6*max_name_score + perc_match*0.4
    ELSE 0
END AS final_score
FROM match
GROUP BY row_key, veridion_id
ORDER BY row_key, final_score DESC
),

ranking AS (
SELECT *,
ROW_NUMBER() OVER (PARTITION BY row_key ORDER BY final_score DESC) as ranking
FROM scores
)

SELECT
  row_key,
  veridion_id,
  round(max_name_score,2) as name_score,
  perc_match,
  round(final_score, 2) as final_score,
  CASE
    WHEN final_score > 99 then 'Exact'
    WHEN final_score > 85 or name_score > 85 then 'High'
    WHEN final_score > 50 or name_score > 70 then 'Medium'
  ELSE 'Low'
  END AS score_category
FROM ranking
WHERE ranking = 1
"""

match_df = duckdb.sql(q).df()

In [ ]:
match_df

## Compare match with input

1. What is missing from match
  - Those that are not matching any country code from input to output - for those we will not match with anything from the list
2. How many low and medium we have
  - 13 low, 90 medium - to be checked manually

In [ ]:
len(match_df), len(input_df)

In [ ]:
q = """
WITH not_match AS (
SELECT *
FROM input_df i
  LEFT JOIN match_df m
  ON i.row_key = m.row_key
WHERE m.row_key IS NULL
),

country_code AS (
SELECT
  input_row_key,
  SUM(input_main_country_code = main_country_code) as same_country
FROM df
GROUP BY input_row_key
HAVING same_country < 1
)

SELECT
  nm.row_key,
  cc.input_row_key
FROM not_match nm
  LEFT JOIN country_code cc
  ON nm.row_key = cc.input_row_key
"""

duckdb.sql(q).df()

In [ ]:
match_df['score_category'].value_counts()

**Get the data**

In [ ]:
print("Additional columns from input: ",set(input_df.columns) - set(df.columns))
print("Additional columns from output: ",set(output_df.columns) - set(df.columns))

In [ ]:
match_df.head()

In [ ]:
input_df_final = df.loc[:, df.columns.str.startswith("input_")].copy()
input_df_final = input_df_final.drop_duplicates()

output_df_final = df.loc[:, ~df.columns.str.startswith("input_") | (df.columns == "input_row_key")].copy()
final_q = """
WITH inputs AS
(SELECT
  *
FROM input_df_final i
  LEFT JOIN match_df m
  ON i.input_row_key = m.row_key
)

SELECT *
FROM inputs i
  LEFT JOIN output_df_final o
  ON i.row_key = o.input_row_key
  AND i.veridion_id = o.veridion_id
"""

final_df = duckdb.sql(final_q).df()

In [ ]:
final_df = final_df.drop_duplicates().reset_index(drop=True)

len(final_df), len(input_df), len(input_df_final)

Illegar characters

In [ ]:
"""
Use AI to identify the illegal characters
"""
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

has_illegal = lambda v: isinstance(v, str) and bool(ILLEGAL_CHARACTERS_RE.search(v))

counts = final_df.map(has_illegal).sum()
print(counts[counts > 0])          # the columns with illegal characters, and how many cells each

# Clean only those columns
for col in counts[counts > 0].index:
    final_df[col] = final_df[col].map(lambda v: ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v)

In [ ]:
final_df["short_description"] = final_df["short_description"].map(
    lambda v: ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v)

final_df.to_excel("output.xlsx", index=False)